# Regression Track — Section A: Dataset & EDA
**Dataset:** Airbnb_Open_Data.csv (regression track only)

This notebook covers rubric Section A (A1–A3): dataset loading & audit, EDA visualisations, and insight commentary. Section B (cleaning, encoding, scaling, feature engineering) is added in a later commit.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
sns.set_palette('colorblind')
pd.set_option('display.max_columns', None)

In [ ]:
# A1: Load dataset
df = pd.read_csv('../../data/regression/Airbnb_Open_Data.csv')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Shape:", df.shape)
df.head()

In [ ]:
# A1: Dtypes and missing value audit
print("Data types:\n", df.dtypes)
print("\nMissing values (sorted):\n", df.isna().sum().sort_values(ascending=False))

**Audit observations:** `license` is almost entirely null and will be dropped in Section B. `price` and `service_fee` are stored as text (e.g. `$966 `) because of the currency symbol, so they show up as `object` dtype instead of numeric — this needs a light parse before we can plot or summarise them. `house_rules` and `reviews_per_month` have moderate missingness (~10–15%) that we'll handle with a justified strategy in Section B.

In [ ]:
# Minimal numeric parse needed purely so we can visualise price/service_fee here.
# Full null-handling, duplicate removal, and outlier treatment happens in Section B (B1).
for col in ['price', 'service_fee']:
    if col in df.columns:
        df[col] = (df[col].astype(str)
                          .str.replace(r'[$,]', '', regex=True)
                          .str.strip()
                          .replace('nan', np.nan)
                          .astype(float))

print(df[['price', 'service_fee']].describe())

In [ ]:
# A1: Target distribution summary (part of the audit)
print("Price summary statistics:")
print(df['price'].describe())
print("\nSkewness (raw):", df['price'].skew().round(3))
print("Skewness (log1p):", np.log1p(df['price']).skew().round(3))

**Target distribution:** `price` ranges widely and is right-skewed (skewness ~shown above), which is typical for price data — most listings cluster at the lower end with a long tail of expensive outliers. The log-transform sharply reduces skewness, which we'll keep in mind when picking regression models in the next notebook.

In [ ]:
# A2: Target distribution plots (raw vs log)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['price'], bins=50, ax=ax[0]).set_title('Price distribution (raw)')
sns.histplot(np.log1p(df['price']), bins=50, ax=ax[1]).set_title('log1p(Price) distribution')
plt.tight_layout()
plt.savefig('price_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# A2: Distribution plots for every other numeric feature (required by rubric A2)
num_cols = df.select_dtypes(include=np.number).columns.tolist()
other_num_cols = [c for c in num_cols if c != 'price']

n = len(other_num_cols)
ncols = 3
nrows = -(-n // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()
for i, col in enumerate(other_num_cols):
    sns.histplot(df[col].dropna(), bins=40, ax=axes[i])
    axes[i].set_title(col)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight')
plt.show()

**Feature distributions:** `minimum_nights` has a long tail with clearly unrealistic values (some listings show minimum stays in the hundreds/thousands of nights) — this will need outlier treatment in Section B. `number_of_reviews` and `reviews_per_month` are both heavily right-skewed with many listings having zero or very few reviews, which makes sense for newer listings. `availability_365` is fairly spread across its full 0–365 range.

In [ ]:
# A2: Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation heatmap (numeric features)')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# A2: Scatter plots — feature vs target (at least two required)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.scatterplot(data=df, x='minimum_nights', y='price', ax=ax[0], alpha=0.3)
ax[0].set_title('Price vs Minimum Nights')
sns.scatterplot(data=df, x='number_of_reviews', y='price', ax=ax[1], alpha=0.3)
ax[1].set_title('Price vs Number of Reviews')
sns.scatterplot(data=df, x='availability_365', y='price', ax=ax[2], alpha=0.3)
ax[2].set_title('Price vs Availability (365)')
plt.tight_layout()
plt.savefig('scatter_price_relationships.png', bbox_inches='tight')
plt.show()

**Heatmap & scatterplot insights:** no single numeric feature is strongly linearly correlated with `price` on its own — correlations are mostly weak, which tells us linear models alone probably won't capture much signal and the tree-based/ensemble models in our algorithm list will likely matter more. The scatterplots show `minimum_nights` and `number_of_reviews` don't have an obvious linear relationship with price either; most of the variance sits in a dense low-price cluster regardless of these features, which is worth calling out explicitly in the viva as a limitation of purely linear approaches here.

In [ ]:
# Bonus: price by category (goes beyond the minimum EDA requirement)
cat_candidates = [c for c in ['room_type', 'neighbourhood_group', 'cancellation_policy'] if c in df.columns]
fig, axes = plt.subplots(1, len(cat_candidates), figsize=(6 * len(cat_candidates), 5))
if len(cat_candidates) == 1:
    axes = [axes]
for i, c in enumerate(cat_candidates):
    sns.boxplot(data=df, x=c, y='price', ax=axes[i])
    axes[i].set_title(f'Price by {c}')
    axes[i].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('price_by_category.png', bbox_inches='tight')
plt.show()

**Category-level insight:** price spread varies noticeably by `room_type` (Entire home/apt skews higher than Private/Shared room, as expected) but looks fairly flat across `cancellation_policy`, suggesting cancellation policy alone won't be a strong price predictor. This is a useful sanity check before we one-hot encode these in Section B.